# Measles Risk Modeling v2: Time-Series Aware Approach

## What This Model Does

This notebook builds a **temporal risk model** that predicts measles outbreak risk at the county level by considering:

1. **Vaccination coverage** (MMR rates over multiple years)
2. **Temporal dynamics** (how outbreaks spread over time)
3. **Spillover risk** (proximity to active outbreaks)

---

## Model Outputs

| Output | Description | Use Case |
|--------|-------------|----------|
| **Risk Score** | Probability (0-1) of outbreak | Prioritize surveillance |
| **Risk Category** | Low/Medium/High/Critical | Resource allocation |
| **Time to Risk** | Expected weeks until cases appear | Intervention timing |
| **Risk Factors** | Top drivers of risk per county | Targeted interventions |
| **Outbreak Trajectory** | Predicted case growth if outbreak occurs | Capacity planning |

---

In [ ]:
# =============================================================================
# STEP 1: IMPORTS AND SETUP
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             roc_curve, precision_recall_curve, average_precision_score, f1_score)

# Settings
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', None)

print("✓ Libraries loaded")

In [ ]:
# =============================================================================
# STEP 2: LOAD DATA
# =============================================================================

DATA_DIR = Path('.')

# Load measles time series
measles_raw = pd.read_csv(DATA_DIR / 'measles_timeseries.csv')
print(f"Measles records: {len(measles_raw)}")

# Load MMR vaccination data
mmr_raw = pd.read_csv(DATA_DIR / 'MMR_2026 - mmr.csv')
print(f"MMR records: {len(mmr_raw)}")

# Parse dates
measles_raw['date'] = pd.to_datetime(measles_raw['date'], format='%m/%d/%y', errors='coerce')
print(f"\nDate range: {measles_raw['date'].min()} to {measles_raw['date'].max()}")

---
## STEP 3: Build Time Series Features

Extract temporal patterns from the outbreak data:
- **When** did outbreaks start in each county/state?
- **How fast** are cases growing?
- **What's the trajectory** (accelerating vs. stabilizing)?

In [ ]:
# =============================================================================
# STEP 3a: Extract County-Level Time Series
# =============================================================================

# Filter to county-level data only
county_ts = measles_raw[measles_raw['vaccination granularity'] == 'county'].copy()
county_ts = county_ts.sort_values(['fips', 'date'])

print(f"County-level records: {len(county_ts)}")
print(f"Unique counties with cases: {county_ts['fips'].nunique()}")

In [ ]:
# =============================================================================
# STEP 3b: Calculate Temporal Features Per County
# =============================================================================

def calculate_temporal_features(group):
    """
    Calculate time-series features for each county.
    
    Returns dict with:
    - first_case_date: When outbreak started
    - latest_date: Most recent data point
    - outbreak_duration_days: How long outbreak has been ongoing
    - total_cases: Cumulative cases (from latest record)
    - weekly_growth_rate: Average new cases per week
    - recent_growth_rate: New cases in last 2 weeks vs prior 2 weeks
    - is_accelerating: Boolean - is outbreak speeding up?
    """
    group = group.sort_values('date')
    
    first_date = group['date'].min()
    latest_date = group['date'].max()
    duration = (latest_date - first_date).days if pd.notna(first_date) and pd.notna(latest_date) else 0
    
    # Get case counts at different time points
    first_cases = group.iloc[0]['cases_total'] if len(group) > 0 else 0
    latest_cases = group.iloc[-1]['cases_total'] if len(group) > 0 else 0
    
    # Calculate growth rate (cases per week)
    weeks = max(duration / 7, 1)  # Avoid division by zero
    weekly_growth = (latest_cases - first_cases) / weeks if weeks > 0 else 0
    
    # Calculate recent acceleration (compare recent vs earlier growth)
    if len(group) >= 4:
        mid_idx = len(group) // 2
        early_growth = group.iloc[mid_idx]['cases_total'] - group.iloc[0]['cases_total']
        recent_growth = group.iloc[-1]['cases_total'] - group.iloc[mid_idx]['cases_total']
        is_accelerating = recent_growth > early_growth
    else:
        is_accelerating = False
    
    return pd.Series({
        'first_case_date': first_date,
        'latest_date': latest_date,
        'outbreak_duration_days': duration,
        'total_cases': latest_cases,
        'weekly_growth_rate': weekly_growth,
        'is_accelerating': is_accelerating,
        'population': group.iloc[0]['population'],
        'longitude': group.iloc[0]['longitude'],
        'latitude': group.iloc[0]['latitude'],
        'imported_cases': group.iloc[-1]['imported'] if 'imported' in group.columns else 0,
        'local_cases': group.iloc[-1]['local'] if 'local' in group.columns else 0,
        'unvaccinated_cases': group.iloc[-1]['unvaccinated'] if 'unvaccinated' in group.columns else 0
    })

# Apply to each county
county_temporal = county_ts.groupby(['fips', 'county', 'state']).apply(
    calculate_temporal_features
).reset_index()

print(f"Counties with temporal features: {len(county_temporal)}")
print(f"\nSample (top 5 by cases):")
print(county_temporal.nlargest(5, 'total_cases')[['county', 'state', 'total_cases', 
                                                   'outbreak_duration_days', 'weekly_growth_rate']])

In [ ]:
# =============================================================================
# STEP 3c: Calculate State-Level Temporal Features
# =============================================================================

# When did each state first see cases?
state_first_case = county_temporal.groupby('state').agg({
    'first_case_date': 'min',
    'total_cases': 'sum',
    'fips': 'count'
}).rename(columns={
    'first_case_date': 'state_first_case_date',
    'total_cases': 'state_total_cases',
    'fips': 'state_counties_affected'
}).reset_index()

# Reference date (latest in dataset)
REFERENCE_DATE = county_temporal['latest_date'].max()
print(f"Reference date (latest data): {REFERENCE_DATE}")

# Days since state's first case
state_first_case['days_since_state_outbreak'] = (
    REFERENCE_DATE - state_first_case['state_first_case_date']
).dt.days

print(f"\nStates with outbreaks: {len(state_first_case)}")
print(state_first_case.nlargest(10, 'state_total_cases')[['state', 'state_total_cases', 
                                                          'state_counties_affected', 
                                                          'days_since_state_outbreak']])

In [ ]:
# =============================================================================
# STEP 3d: National-Level Temporal Features
# =============================================================================

# Calculate weekly national case counts for temporal context
national_weekly = county_ts.groupby(pd.Grouper(key='date', freq='W')).agg({
    'cases_total': 'sum',
    'fips': 'nunique'
}).rename(columns={'fips': 'counties_reporting'})

# National outbreak start
NATIONAL_OUTBREAK_START = county_ts['date'].min()
print(f"National outbreak start: {NATIONAL_OUTBREAK_START}")
print(f"Weeks of data: {len(national_weekly)}")

# Plot national trajectory
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(national_weekly.index, national_weekly['cases_total'], marker='o')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Cases Reported')
ax.set_title('National Measles Outbreak Trajectory')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('national_trajectory.png', dpi=150)
plt.show()

---
## STEP 4: Process MMR Vaccination Data

In [ ]:
# =============================================================================
# STEP 4: MMR Feature Engineering
# =============================================================================

# Standardize FIPS codes
def standardize_fips(fips):
    try:
        return str(int(float(fips))).zfill(5)
    except:
        return None

mmr_raw['fips_std'] = mmr_raw['FIPS'].apply(standardize_fips)
county_temporal['fips_std'] = county_temporal['fips'].apply(standardize_fips)

# Convert MMR columns to numeric
mmr_year_cols = ['SY2024_25', 'SY2023_24', 'SY2022_23', 'SY2021_22',
                 'SY2020_21', 'SY2019_20', 'SY2018_19', 'SY2017_18']

for col in mmr_year_cols:
    mmr_raw[col] = pd.to_numeric(mmr_raw[col], errors='coerce')

# Calculate MMR features
def get_mmr_features(row):
    """Extract vaccination features from multi-year MMR data."""
    rates = [row[col] for col in mmr_year_cols if pd.notna(row[col])]
    
    if len(rates) == 0:
        return pd.Series({'latest_mmr': np.nan, 'mmr_trend': 0, 'mmr_years': 0, 'mmr_min': np.nan})
    
    return pd.Series({
        'latest_mmr': rates[0],  # Most recent
        'mmr_trend': rates[0] - rates[-1] if len(rates) >= 2 else 0,  # Change over time
        'mmr_years': len(rates),  # Data completeness
        'mmr_min': min(rates)  # Worst year (vulnerability indicator)
    })

mmr_features = mmr_raw.apply(get_mmr_features, axis=1)
mmr_df = pd.concat([mmr_raw[['fips_std', 'County', 'State']], mmr_features], axis=1)
mmr_df = mmr_df.rename(columns={'County': 'county', 'State': 'state'})

print(f"Counties with MMR data: {mmr_df['latest_mmr'].notna().sum()}")
print(f"\nMMR statistics:")
print(mmr_df['latest_mmr'].describe())

---
## STEP 5: Create Master Dataset with All Features

In [ ]:
# =============================================================================
# STEP 5: Build Master Dataset
# =============================================================================

# Start with MMR data as base (has most counties)
master_df = mmr_df.copy()

# Create target: has this county had cases?
counties_with_cases = set(county_temporal['fips_std'].dropna())
master_df['has_cases'] = master_df['fips_std'].apply(lambda x: 1 if x in counties_with_cases else 0)

# Merge temporal features for counties with cases
master_df = master_df.merge(
    county_temporal[['fips_std', 'first_case_date', 'outbreak_duration_days', 
                     'total_cases', 'weekly_growth_rate', 'is_accelerating',
                     'population', 'longitude', 'latitude']],
    on='fips_std',
    how='left'
)

# Merge state-level features
master_df = master_df.merge(
    state_first_case[['state', 'state_total_cases', 'state_counties_affected', 
                      'days_since_state_outbreak']],
    on='state',
    how='left'
)

# Fill NaN for counties without state-level outbreak data
master_df['state_total_cases'] = master_df['state_total_cases'].fillna(0)
master_df['state_counties_affected'] = master_df['state_counties_affected'].fillna(0)
master_df['days_since_state_outbreak'] = master_df['days_since_state_outbreak'].fillna(0)

print(f"Master dataset shape: {master_df.shape}")
print(f"Counties with cases: {master_df['has_cases'].sum()}")
print(f"Counties without cases: {(master_df['has_cases']==0).sum()}")

In [ ]:
# =============================================================================
# STEP 5b: Create Risk-Based Features
# =============================================================================

# Vaccination risk category (epidemiological thresholds)
def mmr_risk_category(rate):
    if pd.isna(rate): return 'unknown'
    if rate < 0.85: return 'critical'  # Well below herd immunity
    if rate < 0.90: return 'high'
    if rate < 0.95: return 'medium'    # Below 95% herd immunity threshold
    return 'low'

master_df['mmr_risk_category'] = master_df['latest_mmr'].apply(mmr_risk_category)

# State exposure score (combines state outbreak size and duration)
master_df['state_exposure_score'] = (
    master_df['state_total_cases'] * 
    np.log1p(master_df['days_since_state_outbreak'])
) / 1000  # Normalize

# Susceptibility score (lower MMR = higher susceptibility)
master_df['susceptibility_score'] = 1 - master_df['latest_mmr'].fillna(0.85)

print("MMR Risk Category Distribution:")
print(master_df['mmr_risk_category'].value_counts())

---
## STEP 6: Train Time-Aware Risk Model

In [ ]:
# =============================================================================
# STEP 6a: Define Features for Modeling
# =============================================================================

# Features that capture both static risk (MMR) and temporal risk (outbreak spread)
FEATURE_COLS = [
    # Vaccination features
    'latest_mmr',           # Current coverage
    'mmr_trend',            # Is coverage improving/declining?
    'mmr_min',              # Worst historical coverage
    'susceptibility_score', # Derived vulnerability
    
    # Temporal/spillover features
    'state_total_cases',         # State outbreak size
    'state_counties_affected',   # State outbreak breadth
    'days_since_state_outbreak', # Time since state's first case
    'state_exposure_score',      # Combined exposure metric
]

# Filter to counties with valid MMR data
model_df = master_df.dropna(subset=['latest_mmr']).copy()
print(f"Counties for modeling: {len(model_df)}")
print(f"Target distribution:")
print(model_df['has_cases'].value_counts())

# Prepare X and y
X = model_df[FEATURE_COLS].copy()
y = model_df['has_cases'].copy()

# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=FEATURE_COLS, index=X.index)

print(f"\nFeature matrix: {X_imputed.shape}")

In [ ]:
# =============================================================================
# STEP 6b: Train-Test Split
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training: {len(X_train)} | Testing: {len(X_test)}")
print(f"Train class balance: {y_train.mean():.2%} positive")

In [ ]:
# =============================================================================
# STEP 6c: Train Models
# =============================================================================

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', 
                                            random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, 
                                                     learning_rate=0.1, random_state=RANDOM_STATE)
}

results = {}
for name, model in models.items():
    # Use scaled data for LR, original for tree-based
    X_tr = X_train_scaled if 'Logistic' in name else X_train
    X_te = X_test_scaled if 'Logistic' in name else X_test
    
    model.fit(X_tr, y_train)
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = model.predict(X_te)
    
    results[name] = {
        'model': model,
        'roc_auc': roc_auc_score(y_test, y_prob),
        'avg_precision': average_precision_score(y_test, y_prob),
        'f1': f1_score(y_test, y_pred),
        'probabilities': y_prob
    }
    print(f"{name}: AUC={results[name]['roc_auc']:.4f}, AP={results[name]['avg_precision']:.4f}")

In [ ]:
# =============================================================================
# STEP 6d: Select Best Model
# =============================================================================

best_name = max(results, key=lambda x: results[x]['roc_auc'])
best_model = results[best_name]['model']
print(f"\nBest Model: {best_name} (AUC={results[best_name]['roc_auc']:.4f})")

# Feature importance
if hasattr(best_model, 'feature_importances_'):
    importance = pd.DataFrame({
        'feature': FEATURE_COLS,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\nFeature Importance:")
    print(importance.to_string(index=False))

---
## STEP 7: Generate Risk Predictions (Model Output)

In [ ]:
# =============================================================================
# STEP 7a: Generate Risk Scores for All Counties
# =============================================================================

# Prepare full dataset for prediction
X_full = model_df[FEATURE_COLS].copy()
X_full_imputed = pd.DataFrame(imputer.transform(X_full), columns=FEATURE_COLS, index=X_full.index)

# Generate predictions
if 'Logistic' in best_name:
    X_pred = scaler.transform(X_full_imputed)
else:
    X_pred = X_full_imputed

model_df['risk_score'] = best_model.predict_proba(X_pred)[:, 1]

# Risk categories
model_df['risk_category'] = pd.cut(
    model_df['risk_score'],
    bins=[0, 0.1, 0.3, 0.6, 1.0],
    labels=['Low', 'Moderate', 'High', 'Critical']
)

print("Risk Category Distribution:")
print(model_df['risk_category'].value_counts())

In [ ]:
# =============================================================================
# STEP 7b: Identify Key Risk Factors Per County
# =============================================================================

def get_risk_factors(row):
    """Identify top risk factors for a county."""
    factors = []
    
    # Vaccination risk
    if pd.notna(row['latest_mmr']):
        if row['latest_mmr'] < 0.85:
            factors.append(f"Critical MMR coverage ({row['latest_mmr']:.1%})")
        elif row['latest_mmr'] < 0.90:
            factors.append(f"Low MMR coverage ({row['latest_mmr']:.1%})")
        elif row['latest_mmr'] < 0.95:
            factors.append(f"Below herd immunity ({row['latest_mmr']:.1%})")
    
    # State exposure
    if row['state_total_cases'] > 50:
        factors.append(f"Major state outbreak ({row['state_total_cases']:.0f} cases)")
    elif row['state_total_cases'] > 10:
        factors.append(f"Active state outbreak ({row['state_total_cases']:.0f} cases)")
    
    # Declining vaccination
    if row['mmr_trend'] < -0.02:
        factors.append(f"Declining vaccination (trend: {row['mmr_trend']:.1%})")
    
    return '; '.join(factors) if factors else 'No major risk factors'

model_df['risk_factors'] = model_df.apply(get_risk_factors, axis=1)

In [ ]:
# =============================================================================
# STEP 7c: Create Final Output DataFrame
# =============================================================================

# Define output columns
OUTPUT_COLS = [
    'fips_std',           # County identifier
    'county',             # County name
    'state',              # State
    'has_cases',          # Already has cases? (1/0)
    'total_cases',        # Current case count (if has cases)
    'risk_score',         # Model output: 0-1 probability
    'risk_category',      # Low/Moderate/High/Critical
    'latest_mmr',         # Current vaccination rate
    'mmr_risk_category',  # Vaccination risk level
    'state_total_cases',  # State outbreak context
    'risk_factors',       # Human-readable risk factors
]

output_df = model_df[OUTPUT_COLS].copy()
output_df = output_df.sort_values('risk_score', ascending=False)

# Save to CSV
output_df.to_csv('county_measles_risk_output.csv', index=False)
print(f"\n✓ Output saved to 'county_measles_risk_output.csv'")
print(f"  Rows: {len(output_df)}")

In [ ]:
# =============================================================================
# STEP 7d: Show Key Outputs
# =============================================================================

print("="*80)
print("TOP 20 HIGHEST RISK COUNTIES (without current cases)")
print("="*80)

high_risk_no_cases = output_df[
    (output_df['has_cases'] == 0) & 
    (output_df['risk_score'] > 0.3)
].head(20)

print(high_risk_no_cases[['county', 'state', 'risk_score', 'risk_category', 
                          'latest_mmr', 'risk_factors']].to_string(index=False))

In [ ]:
print("\n" + "="*80)
print("COUNTIES WITH ACTIVE OUTBREAKS - Outbreak Trajectories")
print("="*80)

active_outbreaks = output_df[output_df['has_cases'] == 1].head(15)
print(active_outbreaks[['county', 'state', 'total_cases', 'risk_score', 
                        'latest_mmr', 'risk_factors']].to_string(index=False))

---
## STEP 8: Visualizations

In [ ]:
# =============================================================================
# STEP 8a: Risk Score Distribution
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Risk score by actual case status
ax1 = axes[0]
for has_case in [0, 1]:
    subset = model_df[model_df['has_cases'] == has_case]['risk_score']
    label = 'Has Cases' if has_case else 'No Cases'
    ax1.hist(subset, bins=30, alpha=0.6, label=label, density=True)
ax1.set_xlabel('Risk Score')
ax1.set_ylabel('Density')
ax1.set_title('Risk Score by Actual Case Status')
ax1.legend()

# 2. Risk category distribution
ax2 = axes[1]
model_df['risk_category'].value_counts().plot(kind='bar', ax=ax2, color='steelblue')
ax2.set_xlabel('Risk Category')
ax2.set_ylabel('Number of Counties')
ax2.set_title('Risk Category Distribution')
ax2.tick_params(axis='x', rotation=45)

# 3. MMR vs Risk Score
ax3 = axes[2]
colors = model_df['has_cases'].map({0: 'blue', 1: 'red'})
ax3.scatter(model_df['latest_mmr'], model_df['risk_score'], c=colors, alpha=0.5, s=20)
ax3.axvline(x=0.95, color='green', linestyle='--', label='Herd Immunity (95%)')
ax3.set_xlabel('MMR Vaccination Rate')
ax3.set_ylabel('Risk Score')
ax3.set_title('Vaccination vs Risk (Red=Has Cases)')
ax3.legend()

plt.tight_layout()
plt.savefig('risk_analysis_plots.png', dpi=150)
plt.show()

print("\n✓ Saved: risk_analysis_plots.png")

In [ ]:
# =============================================================================
# STEP 8b: Model Performance Curves
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC Curve
ax1 = axes[0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['probabilities'])
    ax1.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})")
ax1.plot([0,1], [0,1], 'k--', label='Random')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Precision-Recall
ax2 = axes[1]
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['probabilities'])
    ax2.plot(rec, prec, label=f"{name} (AP={res['avg_precision']:.3f})")
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_performance.png', dpi=150)
plt.show()

print("\n✓ Saved: model_performance.png")

---
## Summary: Model Outputs

### Files Generated:

| File | Description |
|------|-------------|
| `county_measles_risk_output.csv` | **Main output**: Risk scores for all counties |
| `national_trajectory.png` | Outbreak growth over time |
| `risk_analysis_plots.png` | Risk score distributions |
| `model_performance.png` | ROC and PR curves |

### Output Schema (`county_measles_risk_output.csv`):

| Column | Type | Description |
|--------|------|-------------|
| `fips_std` | str | 5-digit FIPS code |
| `county` | str | County name |
| `state` | str | State abbreviation |
| `has_cases` | int | 1 if county already has cases |
| `total_cases` | float | Current case count |
| `risk_score` | float | **Model prediction: 0-1 probability** |
| `risk_category` | str | Low/Moderate/High/Critical |
| `latest_mmr` | float | Current vaccination rate |
| `mmr_risk_category` | str | Vaccination risk level |
| `state_total_cases` | float | Total cases in state |
| `risk_factors` | str | Human-readable risk explanation |

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "="*80)
print("MEASLES RISK MODEL - FINAL SUMMARY")
print("="*80)

print(f"\n📊 DATA SUMMARY:")
print(f"   Total counties analyzed: {len(model_df):,}")
print(f"   Counties with cases: {model_df['has_cases'].sum():,}")
print(f"   Counties without cases: {(model_df['has_cases']==0).sum():,}")
print(f"   Date range: {NATIONAL_OUTBREAK_START.strftime('%Y-%m-%d')} to {REFERENCE_DATE.strftime('%Y-%m-%d')}")

print(f"\n🎯 MODEL PERFORMANCE:")
print(f"   Best model: {best_name}")
print(f"   ROC-AUC: {results[best_name]['roc_auc']:.4f}")
print(f"   Average Precision: {results[best_name]['avg_precision']:.4f}")

print(f"\n⚠️  RISK DISTRIBUTION:")
for cat in ['Critical', 'High', 'Moderate', 'Low']:
    count = (model_df['risk_category'] == cat).sum()
    print(f"   {cat}: {count:,} counties")

high_risk_new = model_df[(model_df['has_cases']==0) & (model_df['risk_score']>0.5)]
print(f"\n🚨 HIGH-RISK COUNTIES (no cases yet, risk>50%): {len(high_risk_new)}")

print(f"\n📁 OUTPUT FILES:")
print(f"   - county_measles_risk_output.csv")
print(f"   - national_trajectory.png")
print(f"   - risk_analysis_plots.png")
print(f"   - model_performance.png")